# Experiment 12: Model-Wide Independent Sublayer DBSCAN Tucker Compression

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25]`)  
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj` -> 78 total matrices)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Core Architecture & Innovations:
1. **Dedicated Independent Sublayer Profiling Across All 26 Layers**:
   Eliminates the cross-submodule distribution mismatch by profiling all 3 submodules independently:
   - `act_gate`: Captured at `mlp.act_fn` output ($[B, S, 6912]$, non-linear gating distribution).
   - `act_up`: Captured at `mlp.up_proj` output ($[B, S, 6912]$, linear feature magnitude distribution).
   - `act_down`: Captured at `mlp.down_proj` input ($[B, S, 6912]$, compound element-wise product distribution).
2. **Dedicated DBSCAN Clustering Per Submodule**:
   Each of the 78 projection matrices runs DBSCAN on its own empirical activations, discovering native density-connected coordinate clusters.
3. **Multi-Tier Comparative Sweeps**:
   - **Tier 1: Moderate (77% Sweet Spot)**: `[4, 180, 600]` on all 78 projections (~76–78% error target).
   - **Tier 2: Mixed Optimal**: `gate_proj: [3, 100, 350]` (aggressive regularizing cut), `up_proj: [4, 180, 600]`, `down_proj: [4, 180, 600]` (high-fidelity cuts).
   - **Tier 3: Aggressive Tier**: `[3, 100, 350]` on all 78 projections (direct comparison against Experiment 11 to isolate the independent clustering gain).
4. **Memory-Safe & Zero-VRAM Optimization**:
   All Tucker SVD and Adam GD operations execute on CPU (`device="cpu"`). Model forward passes use `logits_to_keep=1` for ultra-low memory overhead.

In [ ]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    from neural_decomp.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from neural_decomp.utils import get_device_info, save_json_metrics
    print("Loaded neural_decomp library.")
except ImportError:
    from utility import ModelManagementInterface, DeviceMapOptions
    from utility.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from utility.utils import get_device_info, save_json_metrics
    print("Loaded utility library.")

device_info = get_device_info()
print(f"Device: {device_info['device_name']} | CUDA Available: {device_info['cuda_available']}")

In [ ]:
# =====================================================================
# STEP 2: Initialize Model & Tokenizer (Unprocessed Clean Baseline)
# =====================================================================
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_id = "google/gemma-3-1b-it"

mmi = ModelManagementInterface(
    model_id=model_id,
    precision=torch.float32,
    device_map=DeviceMapOptions.AUTO,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

NUM_LAYERS = len(model.model.layers)
D_IN = model.model.layers[0].mlp.gate_proj.weight.shape[1]
D_OUT = model.model.layers[0].mlp.gate_proj.weight.shape[0]

print(f"Loaded {model_id}: {NUM_LAYERS} Transformer Decoder Layers")
print(f"MLP Dimensions: in_features={D_IN}, intermediate_features={D_OUT}")

# Cache original pristine weights on CPU across all 26 layers for calibration & rollback
W_orig_all = {
    l: {
        "gate_proj": model.model.layers[l].mlp.gate_proj.weight.data.clone().cpu(),
        "up_proj":   model.model.layers[l].mlp.up_proj.weight.data.clone().cpu(),
        "down_proj": model.model.layers[l].mlp.down_proj.weight.data.clone().cpu(),
    }
    for l in range(NUM_LAYERS)
}
print(f"Cached pristine weights on CPU for all {NUM_LAYERS} layers (78 projection matrices).")

In [ ]:
# =====================================================================
# STEP 3: Load GLUE MNLI Validation Benchmark
# =====================================================================
ds = load_dataset("nyu-mll/glue", "mnli")["validation_matched"]

label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in label_names]

EVAL_SAMPLE_COUNT = 1000
eval_data = ds.select(range(EVAL_SAMPLE_COUNT))

print(f"Loaded GLUE MNLI: {len(ds):,} total samples | Active Evaluation Subset: {len(eval_data):,} samples")

## Step 1: Simultaneous Tri-Hook Profiling Across All 26 Layers

We register forward hooks on `gate_proj` (`act_fn`), `up_proj`, and `down_proj` input across all 26 layers during baseline inference to record distinct activation trajectories.

In [ ]:
# =====================================================================
# STEP 4: Simultaneous Tri-Hook Profiling & Baseline MNLI Inference
# =====================================================================
layer_trajectories = {
    l: {"gate": [], "up": [], "down": []} for l in range(NUM_LAYERS)
}
current_acts = {
    l: {"gate": None, "up": None, "down": None} for l in range(NUM_LAYERS)
}

def make_hook(layer_idx, sub_key, is_input=False):
    def hook_fn(module, input_tensor, output_tensor):
        t = input_tensor[0] if is_input else (output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor)
        current_acts[layer_idx][sub_key] = t.detach().cpu()
    return hook_fn

hooks = []
for l in range(NUM_LAYERS):
    lmod = model.model.layers[l].mlp
    hooks.append(lmod.act_fn.register_forward_hook(make_hook(l, "gate", is_input=False)))
    hooks.append(lmod.up_proj.register_forward_hook(make_hook(l, "up", is_input=False)))
    hooks.append(lmod.down_proj.register_forward_hook(make_hook(l, "down", is_input=True)))

predictions = []
ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Simultaneous Tri-Hook Profiling & MNLI Inference"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs, logits_to_keep=1)

        for l in range(NUM_LAYERS):
            for sub_key in ["gate", "up", "down"]:
                t = current_acts[l][sub_key]
                if t is not None:
                    pooled = t.squeeze(0).mean(dim=0).numpy()
                    layer_trajectories[l][sub_key].append(pooled)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        predictions.append(pred_label)
        ground_truth.append(sample["label"])

for h in hooks:
    h.remove()

acts_all = {
    l: {
        "gate": np.stack(layer_trajectories[l]["gate"]),
        "up":   np.stack(layer_trajectories[l]["up"]),
        "down": np.stack(layer_trajectories[l]["down"]),
    }
    for l in range(NUM_LAYERS)
}

baseline_accuracy = accuracy_score(ground_truth, predictions)
print(f"\nUncompressed Baseline Accuracy: {baseline_accuracy * 100:.2f}%")
print(f"Successfully captured tri-hook activation matrices across all {NUM_LAYERS} layers (78 profiles).")

## Step 2: Optimization Utility: Tucker with Adam Gradient Descent

We define `optimize_tucker_gd` which initializes factor matrices and core via SVD, followed by 35 steps of PyTorch Adam gradient descent refinement on CPU.

In [ ]:
# =====================================================================
# STEP 5: Define Tucker Decomposition with Adam Gradient Descent
# =====================================================================
def optimize_tucker_gd(T, ranks, num_steps=35, lr=1e-3, device="cpu"):
    core_init, factors_init = tucker(T, rank=ranks, init='svd')
    core_param = torch.nn.Parameter(core_init.clone().to(device))
    factors_param = [torch.nn.Parameter(f.clone().to(device)) for f in factors_init]
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)
    T_target = T.to(device)

    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param)).cpu()
        final_err = (torch.norm(T.cpu() - T_recon_final) / torch.norm(T.cpu())).item()

    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, final_err

print("Tucker Adam GD optimizer defined.")

## Step 3: Pre-Clustering Submodules Independently Across All 26 Layers

For every layer $l \in [0 \dots 25]$, each of `gate_proj`, `up_proj`, and `down_proj` independently runs DBSCAN to extract its own top 6 uniform slices ($2,400$ coords) and isolate superweights.

In [ ]:
# =====================================================================
# STEP 6: Independent Pre-Clustering for All 78 Submodules
# =====================================================================
CHUNK_SIZE = 400
NUM_CHUNKS = 6

def cluster_submodule(acts_matrix, weight_tensor, is_col=False):
    v = np.mean(acts_matrix, axis=0)
    std_v = np.std(v)
    eps = max(0.04, float(std_v * 0.18))

    db = DBSCAN(eps=eps, min_samples=30, metric="euclidean")
    labels = db.fit_predict(v.reshape(-1, 1))

    max_mags = np.max(np.abs(acts_matrix), axis=0)
    variances = np.var(acts_matrix, axis=0)
    super_mask = (labels == -1) | (max_mags > 3.0) | (variances >= np.quantile(variances, 0.99))
    super_indices = np.where(super_mask)[0]

    unique_labels = [lab for lab in np.unique(labels) if lab != -1]

    chunk_list = []
    for lab in unique_labels:
        c_idx = np.where((labels == lab) & (~super_mask))[0]
        if len(c_idx) == 0:
            continue
        sorted_idx = c_idx[np.argsort(v[c_idx])]
        num_full = len(sorted_idx) // CHUNK_SIZE
        for ci in range(num_full):
            chunk_list.append(sorted_idx[ci * CHUNK_SIZE : (ci + 1) * CHUNK_SIZE])
            if len(chunk_list) >= NUM_CHUNKS:
                break
        if len(chunk_list) >= NUM_CHUNKS:
            break

    if len(chunk_list) < NUM_CHUNKS:
        assigned = set(np.concatenate(chunk_list) if chunk_list else [])
        avail = [i for i in range(len(v)) if i not in assigned and not super_mask[i]]
        needed = NUM_CHUNKS - len(chunk_list)
        for ci in range(needed):
            if len(avail) >= CHUNK_SIZE:
                chunk_list.append(np.array(avail[:CHUNK_SIZE]))
                avail = avail[CHUNK_SIZE:]

    active_coords = np.concatenate(chunk_list)

    if is_col:
        T = torch.stack([weight_tensor[:, c].T.float().cpu() for c in chunk_list], dim=0)
    else:
        T = torch.stack([weight_tensor[c, :].float().cpu() for c in chunk_list], dim=0)

    return {
        "tensor": T,
        "chunk_list": chunk_list,
        "active_coords": active_coords,
        "super_indices": super_indices,
        "is_col": is_col,
    }

print("Running independent DBSCAN clustering across all 26 layers...")
layer_submodule_data = {}

for l in range(NUM_LAYERS):
    layer_submodule_data[l] = {
        "gate_proj": cluster_submodule(acts_all[l]["gate"], W_orig_all[l]["gate_proj"], is_col=False),
        "up_proj":   cluster_submodule(acts_all[l]["up"],   W_orig_all[l]["up_proj"],   is_col=False),
        "down_proj": cluster_submodule(acts_all[l]["down"], W_orig_all[l]["down_proj"], is_col=True),
    }

print(f"Pre-clustering complete for all {NUM_LAYERS} layers (78 independent clusterings).")

## Step 4: Multi-Tier Model-Wide Compression & Downstream Evaluation (1k Samples)

We evaluate the full model across 5 distinct compression tiers on 1,000 GLUE MNLI samples to determine the optimal trade-off:
1. **Tier 1: Ultra-High Fidelity (`[5, 280, 850]`)**: ~58% error target (~37.7M params cut).
2. **Tier 2: High Fidelity (`[5, 240, 750]`)**: ~64% error target (~70.6M params cut).
3. **Tier 3: Balanced Tier (`[4, 250, 800]`)**: ~70% error target (~73.6M params cut).
4. **Tier 4: Moderate-73 (`[4, 220, 700]`)**: ~73% error target (~97.8M params cut).
5. **Tier 5: Legacy 77% Tier (`[4, 180, 600]`)**: ~77% error target (~122.4M params cut).


In [ ]:
# =====================================================================
# STEP 7: Execute Multi-Tier Compression Loops & GLUE MNLI Evaluation
# =====================================================================
eval_tiers = [
    {
        "name": "Tier 1: Ultra-High Fidelity ([5, 280, 850])",
        "ranks": {
            "gate_proj": [5, 280, 850],
            "up_proj":   [5, 280, 850],
            "down_proj": [5, 280, 850],
        }
    },
    {
        "name": "Tier 2: High Fidelity ([5, 240, 750])",
        "ranks": {
            "gate_proj": [5, 240, 750],
            "up_proj":   [5, 240, 750],
            "down_proj": [5, 240, 750],
        }
    },
    {
        "name": "Tier 3: Balanced Tier ([4, 250, 800])",
        "ranks": {
            "gate_proj": [4, 250, 800],
            "up_proj":   [4, 250, 800],
            "down_proj": [4, 250, 800],
        }
    },
    {
        "name": "Tier 4: Moderate-73 ([4, 220, 700])",
        "ranks": {
            "gate_proj": [4, 220, 700],
            "up_proj":   [4, 220, 700],
            "down_proj": [4, 220, 700],
        }
    },
    {
        "name": "Tier 5: Legacy 77% Tier ([4, 180, 600])",
        "ranks": {
            "gate_proj": [4, 180, 600],
            "up_proj":   [4, 180, 600],
            "down_proj": [4, 180, 600],
        }
    },
]

tier_benchmarks = []

for tier in eval_tiers:
    tier_name = tier["name"]
    tier_ranks = tier["ranks"]
    
    print(f"\n{'='*95}")
    print(f"Running Full-Model Evaluation for: {tier_name}")
    print(f"{'='*95}")
    
    tier_gate_errs, tier_up_errs, tier_down_errs = [], [], []
    total_params_saved = 0
    
    # 1. Factorize and inject across all 26 layers
    for l in range(NUM_LAYERS):
        lmod = model.model.layers[l].mlp
        sdata_layer = layer_submodule_data[l]
        
        for sub_name in ["gate_proj", "up_proj", "down_proj"]:
            sdata = sdata_layer[sub_name]
            ranks = tier_ranks[sub_name]
            
            cg, fg, T_recon, err = optimize_tucker_gd(
                sdata["tensor"], ranks=ranks, num_steps=35, lr=1e-3, device="cpu"
            )
            
            if sub_name == "gate_proj": tier_gate_errs.append(err)
            elif sub_name == "up_proj":  tier_up_errs.append(err)
            elif sub_name == "down_proj": tier_down_errs.append(err)
            
            orig_p = sdata["tensor"].numel()
            comp_p = cg.numel() + sum(f.numel() for f in fg)
            total_params_saved += (orig_p - comp_p)
            
            # Live injection
            mod_ref = getattr(lmod, sub_name)
            orig_w = W_orig_all[l][sub_name]
            mod_ref.weight.data = orig_w.clone().to(model.device)
            
            if sdata["is_col"]:
                for k, c in enumerate(sdata["chunk_list"]):
                    mod_ref.weight.data[:, c] = T_recon[k].T.to(device=model.device, dtype=mod_ref.weight.dtype)
                mod_ref.weight.data[:, sdata["super_indices"]] = orig_w[:, sdata["super_indices"]].to(model.device)
            else:
                for k, c in enumerate(sdata["chunk_list"]):
                    mod_ref.weight.data[c, :] = T_recon[k].to(device=model.device, dtype=mod_ref.weight.dtype)
                mod_ref.weight.data[sdata["super_indices"], :] = orig_w[sdata["super_indices"], :].to(model.device)
                
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    mean_gate_err = np.mean(tier_gate_errs) * 100
    mean_up_err   = np.mean(tier_up_errs) * 100
    mean_down_err = np.mean(tier_down_errs) * 100
    
    print(f"Layer Factorization Complete. Mean Recon Errors: gate={mean_gate_err:.1f}%, up={mean_up_err:.1f}%, down={mean_down_err:.1f}%")
    print(f"Total Parameters Eliminated: {total_params_saved:,}")
    
    # 2. Evaluate on GLUE MNLI
    preds, gts = [], []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=f"Evaluating {tier_name}"):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs, logits_to_keep=1)

            next_token_logits = outputs.logits[0, -1, :]
            candidate_logits = next_token_logits[label_token_ids]
            pred_label = torch.argmax(candidate_logits).item()

            preds.append(pred_label)
            gts.append(sample["label"])

    acc = accuracy_score(gts, preds)
    delta = acc - baseline_accuracy
    
    print(f"\nResult for {tier_name}:")
    print(f"  Downstream Accuracy: {acc * 100:.2f}% (Δ vs Baseline: {delta * 100:+.2f}%)")
    print(f"  Total Params Cut:    {total_params_saved:,}")
    
    tier_benchmarks.append({
        "Variant": tier_name,
        "Mean_Gate_Err": round(mean_gate_err, 2),
        "Mean_Up_Err": round(mean_up_err, 2),
        "Mean_Down_Err": round(mean_down_err, 2),
        "Params_Eliminated": total_params_saved,
        "Accuracy": round(acc * 100, 2),
        "Delta": round(delta * 100, 2),
    })

# Restore pristine model weights across all 26 layers
for l in range(NUM_LAYERS):
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        getattr(model.model.layers[l].mlp, sub_name).weight.data = W_orig_all[l][sub_name].clone().to(model.device)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nRestored all 26 layers to pristine weights.")

## Step 5: Comparative Synthesis & Metrics Export

We tabulate the multi-tier performance comparisons against baseline and export metrics to `artifacts/12_all_layers_independent_sublayer_results.json`.

In [ ]:
# =====================================================================
# STEP 8: Comparative Synthesis & Artifact Export
# =====================================================================
total_mlp_params_orig = NUM_LAYERS * 3 * (D_IN * D_OUT)  # 621,084,672

print("=" * 115)
print(f"{'Variant':<55} | {'Gate Err':<9} | {'Up Err':<8} | {'Down Err':<9} | {'Params Cut':<11} | {'Accuracy':<9} | {'Delta':<8}")
print("=" * 115)
print(f"{'Baseline (Uncompressed)':<55} | {'0.00%':<9} | {'0.00%':<8} | {'0.00%':<9} | {'0':<11} | {baseline_accuracy*100:>7.2f}% | {'+0.00%':<8}")

for r in tier_benchmarks:
    print(f"{r['Variant']:<55} | {r['Mean_Gate_Err']:>6.2f}% | {r['Mean_Up_Err']:>5.2f}% | {r['Mean_Down_Err']:>6.2f}% | {r['Params_Eliminated']:>10,} | {r['Accuracy']:>7.2f}% | {r['Delta']:>+6.2f}%")

print("=" * 115)

os.makedirs("artifacts", exist_ok=True)
output_payload = {
    "experiment": "12_all_layers_independent_sublayer_dbscan",
    "target_model": model_id,
    "num_layers": NUM_LAYERS,
    "num_projections": NUM_LAYERS * 3,
    "total_mlp_params_orig": total_mlp_params_orig,
    "baseline_accuracy": round(baseline_accuracy * 100, 2),
    "tiers": tier_benchmarks,
    "timings": NOTEBOOK_TIMINGS,
}

with open("artifacts/12_all_layers_independent_sublayer_results.json", "w") as f:
    json.dump(output_payload, f, indent=2)

print(f"Saved benchmark results to artifacts/12_all_layers_independent_sublayer_results.json")